# HuggingFace API Key
## 모델 불러오기

In [1]:
import os
from dotenv import load_dotenv

# dotenv 파일에서 환경변수 로드
load_dotenv()

True

In [2]:
# API 키 확인
api_key = os.getenv("HF_TOKEN")
if api_key:
    print("Hugging Face API 키가 설정되었습니다.")
else:
    print("Hugging Face API 키가 없습니다.")

Hugging Face API 키가 설정되었습니다.


### device 정의

In [8]:
import torch

# GPU 있으면 GPU, 없으면 CPU
device = 0 if torch.cuda.is_available() else -1

device

-1

### 창작용(Creative Writing)

In [13]:
from langchain_huggingface import HuggingFacePipeline

# 창의적인 생성에 적합한 설정
# 텍스트 생성 모델
llm_creative = HuggingFacePipeline.from_model_id(
    model_id = "LGAI-EXAONE/EXAONE-4.0-1.2B", # 모델 ID
    task = "text-generation",       # 텍스트 생성 작업 지정
    device = device,                # GPU 사용 설정 
    model_kwargs = dict(
        cache_dir = "./models/"     # 모델이 저장될 폴더 지정
    ),
    pipeline_kwargs = dict(
        top_p = 0.9,                # 확률 분포 상위 90%만 사용 : 너무 이상한 토큰은 없앰
        max_new_tokens = 512,       # 긴 생성 허용
        do_sample = True,           # 샘플링 활성화 : 확률 분포에서 랜덤 샘플링(매번 결과가 달라짐)
        repetition_penalty=1.05     # 약간의 반복 제어
    )
)

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [14]:
from langchain_huggingface import ChatHuggingFace

creative_llm = ChatHuggingFace(llm=llm_creative) 
# wrapping(랩핑) -> 채팅 인터페이스로 변환하는 wrapper
# HuggingFacePipeline -> 엔진 / ChatHuggingFace -> 자동차(핸들+인터페이스)
# 모델을 다시 만드는게 아니라 Chat 형태로 바꿔주는 어댑터
# 예시) llm.invoke([("system", "너는 시인이다"),("human", "시 써줘")])

In [15]:
prompt = "시간 여행자가 조선을 방문하는 5문장의 짧은 소설을 써줘."

# 사용자 입력에 대한 응답 생성
# response는 AI가 생성한 이야기
response = creative_llm.invoke(prompt)

print(response.content)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[|user|]
시간 여행자가 조선을 방문하는 5문장의 짧은 소설을 써줘.[|endofturn|]
[|assistant|]
<think>

</think>

**"조선의 마지막 letter"**  

1. 청군의 침입 소식이 전해지자, 조선의 미래인 18세기 한명당 300명의 시간여행자들이 비상 소집되었다. 그들은 왕의 명으로만 가능한 비밀 프로젝트였다.  

2. 역사 기록에 없는 인물들—검은 머리의 학자, 독신과 결혼한 여성, 개종된 양반—이 서로 만나게 된다. 과거 시험 대신 현대식 편지를 쓰며 고민한다.  

3. 그 중 한 소년이 갑자기 알게 된 조선의 지도를 본 채로, 현대의 스마트폰을 들고 있음을 깨닫는다. 과거와 현재의 경계에서 혼란스러워한다.  

4. 어느 날, 그는 자신이 미래의 희생양이었음을 직감한다. 왕이 그들을 구하기 위해 한 편의 시를 작성해 달라고 부탁받는다.  

5. 마지막 줄엔 눈물 섞인 편지가 놓여 있었다. "만약 네가 살아있다면…" 한 글자 뒤엔 미래의 현재가 숨어 있었다.


In [31]:
# assistant태그 이후 텍스트만 추출
def extract_assistant_response(text):
    """assistant 태그 이후의 텍스트만 추출"""
    import re # 정규 표현식 모듈

    pattern = r'\[\|assistant\|\]\s*(.*?)(?:\[\|endofturn\|\]|$)'
    match = re.search(pattern, text, re.DOTALL)
    extracted = match.group(1).strip() if match else ""

    # think 블록 제거
    cleaned = re.sub(r'<think>.*?</think>', '', extracted, flags=re.DOTALL)

    # 기타 정리
    cleaned = re.sub(r'\*{3,}', '', cleaned)

    return cleaned.strip()
    

In [32]:
result = extract_assistant_response(response.content)
print(result)

지구에서 가장 깊은 바다는 **해령(Mid-Ocean Ridge)**이 형성하는 중앙 해령 지역입니다. 대표적인 예로는 다음과 같은 곳들이 있습니다:  

1. **중앙 해령**  
   - 대서양 중앙 해령(Mid-Atlantic Ridge): 약 6,900만 km 깊이의 해구와 함께 활동 중입니다.  
   - 태평양 중앙 해령(Pacific Rise): 마리아나 해구와 연결되어 있으며, 판 경계면에서 화산 활동이 활발합니다.  

2. **해구(Subduction Zone)**  
   - 태평양판이 다른 판 아래로 섭입하며 형성된 deepest 해구는 마리아나 해구(약 11,000km 깊이)가 있습니다.  

3. **심해 열곡(Seamounts)**  
   - 해령 주변의 작은 화산섬으로, 깊이는 다양하지만 일반적으로 해구보다 얕습니다.  

현재까지 확인된 최대 깊이는 마리아나 해구의 해구 바닥(약 10,984km)이지만, 해령에서도 마


### 정확한 답변용(Fact-based)

In [22]:
from langchain_huggingface import HuggingFacePipeline

# 창의적인 생성에 적합한 설정
# 텍스트 생성 모델
llm_fact = HuggingFacePipeline.from_model_id(
    model_id = "LGAI-EXAONE/EXAONE-4.0-1.2B", # 모델 ID
    task = "text-generation",       # 텍스트 생성 작업 지정
    device = device,                # GPU 사용 설정 
    model_kwargs = dict(
        cache_dir = "./models/"     # 모델이 저장될 폴더 지정
    ),
    pipeline_kwargs = dict(
        temperature = 0.1,          # 무작위성 낮춤(일관된 결과)
        top_p = 1.0,                # 모든 확률 고려(안정성 우선)
        max_new_tokens = 256,       # 짧고 정확한 응답
        do_sample = False,          # 샘플링 비활성화 
        repetition_penalty=1.05     # 약간의 반복 제어
    )
)

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'max_new_tokens', 'repetition_penalty', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [26]:
from langchain_huggingface import ChatHuggingFace

fact_llm = ChatHuggingFace(llm=llm_fact) 
# wrapping(랩핑) -> 채팅 인터페이스로 변환하는 wrapper
# HuggingFacePipeline -> 엔진 / ChatHuggingFace -> 자동차(핸들+인터페이스)
# 모델을 다시 만드는게 아니라 Chat 형태로 바꿔주는 어댑터
# 예시) llm.invoke([("system", "너는 시인이다"),("human", "시 써줘")])

In [34]:
prompt = "지구에서 가장 깊은 바다는 어디야?"

# 사용자 입력에 대한 응답 생성
# response는 AI가 생성한 이야기
response = fact_llm.invoke(prompt)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [35]:
result = extract_assistant_response(response.content)
print(result)

지구에서 가장 깊은 바다는 **해령(Mid-Ocean Ridge)**이 형성하는 중앙 해령 지역입니다. 대표적인 예로는 다음과 같은 곳들이 있습니다:  

1. **중앙 해령**  
   - 대서양 중앙 해령(Mid-Atlantic Ridge): 약 6,900만 km 깊이의 해구와 함께 활동 중입니다.  
   - 태평양 중앙 해령(Pacific Rise): 마리아나 해구와 연결되어 있으며, 판 경계면에서 화산 활동이 활발합니다.  

2. **해구(Subduction Zone)**  
   - 태평양판이 다른 판 아래로 섭입하며 형성된 deepest 해구는 마리아나 해구(약 11,000km 깊이)가 있습니다.  

3. **심해 열곡(Seamounts)**  
   - 해령 주변의 작은 화산섬으로, 깊이는 다양하지만 일반적으로 해구보다 얕습니다.  

현재까지 확인된 최대 깊이는 마리아나 해구의 해구 바닥(약 10,984km)이지만, 해령에서도 마
